In [1]:
import torch
import torch.nn as nn

class HoloNetVault(nn.Module):
    def __init__(self, d_model, rank=16):
        super().__init__()
        self.d_model = d_model

        # 1. Cayley Transform Parameter (Learnable Skew-Symmetric generator)
        # Initialized tiny so the rotation starts very close to the Identity matrix
        self.S_params = nn.Parameter(torch.randn(d_model, d_model) * 0.01)

        # 2. Learnable Decay Factor (\gamma)
        # Starts high to retain memory, but allows the model to learn to forget noise
        self.gamma = nn.Parameter(torch.ones(1) * 2.0) # sigmoid(2.0) ≈ 0.88

        # 3. The Low-Rank Bottleneck (L = A * B^T)
        self.A = nn.Parameter(torch.randn(d_model, rank) / (d_model ** 0.5))
        self.B = nn.Parameter(torch.randn(rank, d_model) / (rank ** 0.5))

        # 4. Input-Dependent Gating (Strictly Associative!)
        # g_t ONLY looks at x_t. This ensures parallel scan compatibility.
        self.gate_proj = nn.Linear(d_model, d_model)

        # 5. Information Injection
        self.W_in = nn.Linear(d_model, d_model)

    def get_rotation_matrix(self):
        """ Computes the stable orthogonal matrix D using the Cayley Transform """
        # Force S to be strictly skew-symmetric: S = (P - P^T) / 2
        S = (self.S_params - self.S_params.T) / 2
        I = torch.eye(self.d_model, device=S.device)

        # D = (I - S)(I + S)^-1
        # Using torch.linalg.solve(A, B) computes A^-1 B safely and efficiently
        D = torch.linalg.solve(I + S, I - S)
        return D

    def forward(self, x_seq):
        """
        x_seq: (Batch, Seq_Len, D_Model)
        """
        batch_size, seq_len, _ = x_seq.shape
        device = x_seq.device

        # Initialize the Vault
        h_t = torch.zeros(batch_size, self.d_model, device=device)

        # Get the stable rotation matrix for this forward pass
        D = self.get_rotation_matrix()

        # Clamp gamma smoothly between 0 and 1 using sigmoid
        gamma_decay = torch.sigmoid(self.gamma)

        output_states = []

        # Note: This is the sequential unoptimized prototype for testing the math.
        # In a production distributed environment, this loop is replaced by an Associative Scan.
        for t in range(seq_len):
            x_t = x_seq[:, t, :]

            # 1. Calculate Associative Gate (depends ONLY on input)
            g_t = torch.sigmoid(self.gate_proj(x_t))

            # 2. Low-Rank Projection
            low_rank_update = (h_t @ self.B.T) @ self.A.T

            # 3. The Grand Equation: Decay + Rotation + Gated Low-Rank + New Info
            h_t = gamma_decay * (h_t @ D.T) + (g_t * low_rank_update) + self.W_in(x_t)

            output_states.append(h_t)

        return torch.stack(output_states, dim=1)

# --- Sanity Check ---
if __name__ == "__main__":
    model = HoloNetVault(d_model=512, rank=16).cuda()
    dummy_input = torch.randn(4, 100, 512).cuda() # Batch 4, Seq 100, Dim 512
    out = model(dummy_input)
    print(f"HoloNet Vault Output Shape: {out.shape}")
    print("Cayley Transform & Associative Math Compiled Successfully! 🚀")

HoloNet Vault Output Shape: torch.Size([4, 100, 512])
Cayley Transform & Associative Math Compiled Successfully! 🚀


In [2]:
class HoloNetBlock(nn.Module):
    def __init__(self, d_model, n_heads, vault_rank=16, vault_dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.vault_dropout = vault_dropout

        # 1. The Fast Working Memory (Sniper)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            batch_first=True
        )
        self.ln1 = nn.LayerNorm(d_model)

        # 2. The Persistent Global Memory (The Engine you just built!)
        self.vault = HoloNetVault(d_model=d_model, rank=vault_rank)
        self.ln_vault = nn.LayerNorm(d_model)

        # 3. The Output Processing (Feed Forward / MLP)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        """ x shape: (Batch, Seq_Len, D_Model) """
        device = x.device
        seq_len = x.shape[1]

        # --- A. Local Attention (The Sniper) ---
        # Generate a causal mask so the model can't cheat by looking at future tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(device)

        attn_out, _ = self.attention(x, x, x, attn_mask=causal_mask)

        # 🚨 THE KILL SWITCH (Vault-Forcing Dropout) 🚨
        # If we are training, blind the Sniper 20% of the time!
        if self.training and torch.rand(1).item() < self.vault_dropout:
            attn_out = torch.zeros_like(attn_out)

        # Add residual and normalize
        x = self.ln1(x + attn_out)

        # --- B. The Global Vault (Semantic Tracking) ---
        # Run the sequence through the Cayley-stabilized recurrent vault
        vault_out = self.vault(x)
        x = x + self.ln_vault(vault_out)

        # --- C. Final Feed-Forward Processing ---
        ffn_out = self.ffn(x)
        out = self.ln2(x + ffn_out)

        return out

# --- Sanity Check Phase 2 ---
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize the complete Hybrid Block
    hybrid_layer = HoloNetBlock(d_model=512, n_heads=8).to(device)

    # Dummy sequence (e.g., 4 sentences, 100 words each, 512 embedding size)
    dummy_x = torch.randn(4, 100, 512).to(device)

    # Run the hybrid architecture
    final_output = hybrid_layer(dummy_x)

    print(f"HoloNet Hybrid Layer Output Shape: {final_output.shape}")
    print("Kill Switch & Attention Handoff Executed Successfully! 🎯")

HoloNet Hybrid Layer Output Shape: torch.Size([4, 100, 512])
Kill Switch & Attention Handoff Executed Successfully! 🎯


In [3]:
import torch.optim as optim
import torch.nn.functional as F

# --- The Colab Training Arena ---
print("Initializing HoloNet Training Sequence... 🚀")

# 1. Setup a mini LLM environment
vocab_size = 1000
seq_len = 50
batch_size = 8
d_model = 512

# We add a simple embedding layer and output head to our HoloNetBlock
class MiniHoloNetLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.holonet_block = HoloNetBlock(d_model=d_model, n_heads=8, vault_dropout=0.2)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.holonet_block(x)
        return self.lm_head(x)

model = MiniHoloNetLLM().to(device)

# 2. THE DUAL LEARNING RATE STRATEGY (Reviewer #2's demand)
# We separate the Cayley S_params from the rest of the network
cayley_params = [model.holonet_block.vault.S_params]
base_params = [p for n, p in model.named_parameters() if "S_params" not in n]

optimizer = optim.AdamW([
    {'params': base_params, 'lr': 1e-3},         # Fast learning for Attention/Linear
    {'params': cayley_params, 'lr': 1e-4, 'weight_decay': 0.0} # Slow, safe learning for Rotation
])

# 3. Create a repeatable dummy task (predicting the next number in a sequence)
x_data = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)
y_target = torch.randint(0, vocab_size, (batch_size, seq_len)).to(device)

# 4. The Training Loop
epochs = 50
print("Starting Training Loop...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    logits = model(x_data)

    # Calculate Loss (Cross Entropy for Language Modeling)
    # Reshape logits to (Batch * Seq_Len, Vocab_Size) to match targets
    loss = F.cross_entropy(logits.view(-1, vocab_size), y_target.view(-1))

    # Backward pass & Optimize
    loss.backward()

    # Gradient clipping to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:02d}/{epochs} | Loss: {loss.item():.4f}")

print("Training Complete! The Vault is alive. 🧠")

Initializing HoloNet Training Sequence... 🚀
Starting Training Loop...
Epoch 00/50 | Loss: 7.0513
Epoch 10/50 | Loss: 0.7969
Epoch 20/50 | Loss: 0.0176
Epoch 30/50 | Loss: 0.0042
Epoch 40/50 | Loss: 0.0015
Epoch 49/50 | Loss: 0.0028
Training Complete! The Vault is alive. 🧠
